In [4]:
import torch

def check_m4_gpu():
    if torch.backends.mps.is_available():
        print("✅ M4 GPU (Metal) is AVAILABLE via MPS.")
        # Create a small tensor to verify
        device = torch.device("mps")
        x = torch.ones(1, device=device)
        print(f"Test tensor successfully moved to: {x.device}")
    else:
        print("❌ M4 GPU is NOT available to PyTorch. Check your macOS version (12.3+) or PyTorch installation.")

if __name__ == "__main__":
    check_m4_gpu()

✅ M4 GPU (Metal) is AVAILABLE via MPS.
Test tensor successfully moved to: mps:0


In [7]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset" # Adjust if your local path differs
TRAIN_PNG_SOURCE = os.path.join(BASE_PATH, "train/png")
ANNOTATIONS_FILE = os.path.join(BASE_PATH, "train/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# Initialize directories
for c_type in CHART_TYPES:
    os.makedirs(os.path.join(BASE_PATH, f"{c_type}_plots/train/png"), exist_ok=True)

def copy_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(TRAIN_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/train/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_with_m4_cores():
    with open(ANNOTATIONS_FILE, 'r') as f:
        data = json.load(f)

    print(f"Starting split using 10-core parallel processing...")
    counts = {c: 0 for c in CHART_TYPES}
    
    # Using ThreadPoolExecutor to handle I/O bound copying across cores
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\nSplit Summary:")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    split_with_m4_cores()

Starting split using 10-core parallel processing...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157070/157070 [00:28<00:00, 5430.29it/s]



Split Summary:
- vbar_categorical: 52463 images
- hbar_categorical: 52700 images
- dot_line: 26010 images
- line: 25897 images


In [15]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION FOR VAL ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset" 
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# Initialize validation directories
for c_type in CHART_TYPES:
    os.makedirs(os.path.join(BASE_PATH, f"{c_type}_plots/val/png"), exist_ok=True)

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_with_m4():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ Validation annotations not found at: {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"Starting Validation split using 10-core parallel processing...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\nValidation Split Summary:")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    split_val_with_m4()

❌ Validation annotations not found at: /Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset/val/annotations.json


In [18]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset"

# PlotQA validation usually sits in its own 'val' subfolder
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# --- INITIALIZATION ---
def init_val_dirs():
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print("✅ Validation directory structure ready.")

# --- PROCESSING ---
def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type'] # vbar_categorical, hbar_categorical, dot_line, line
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_with_m4():
    # Final check for file existence
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ ERROR: Cannot find validation annotations at: {VAL_ANNOTATIONS}")
        print("Please ensure you have downloaded the Validation Annotations from the official PlotQA links.")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Starting Validation Split (Parallel I/O on 10 Cores)...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_with_m4()

✅ Validation directory structure ready.
❌ ERROR: Cannot find validation annotations at: /Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset/val/annotations.json
Please ensure you have downloaded the Validation Annotations from the official PlotQA links.


In [21]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION (Updated from your screenshot) ---
# Your screenshot shows: Documents > Documents > Projects...
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

def init_val_dirs():
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print("✅ Validation specialized directories initialized.")

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_dataset():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ STILL NOT FOUND: {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Processing {len(data)} validation entries using 10 M4 Cores...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_dataset()

✅ Validation specialized directories initialized.
🚀 Processing 33650 validation entries using 10 M4 Cores...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33650/33650 [00:04<00:00, 6757.63it/s]



--- Validation Split Summary ---
- vbar_categorical: 11240 images
- hbar_categorical: 11292 images
- dot_line: 5571 images
- line: 5547 images


In [1]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION (Directly from your screenshot) ---
# This absolute path prevents folders from being created in your 'notebooks' section
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

def init_val_dirs():
    # This ensures the new folders are created inside the PlotQA-Dataset directory, not the notebooks folder
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print(f"✅ Specialized directories initialized at: {BASE_PATH}")

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_dataset():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ ERROR: Cannot find annotations at {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Processing {len(data)} entries from PlotQA Validation set...")
    counts = {c: 0 for c in CHART_TYPES}
    
    # Using your 10-core CPU for maximum I/O speed
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_dataset()

✅ Specialized directories initialized at: /Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset
🚀 Processing 33650 entries from PlotQA Validation set...


100%|███████████████████████████████████████████████████████████████████| 33650/33650 [00:04<00:00, 7998.51it/s]



--- Validation Split Summary ---
- vbar_categorical: 11240 images
- hbar_categorical: 11292 images
- dot_line: 5571 images
- line: 5547 images


In [2]:
import json
import os
from tqdm import tqdm

# --- CONFIGURATION (Exact M4 Path) ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# Original Source Annotations
SOURCE_ANN = {
    "train": os.path.join(BASE_PATH, "train/annotations.json"),
    "val": os.path.join(BASE_PATH, "val/annotations.json")
}

def generate_specialized_metadata():
    for split in ['train', 'val']:
        ann_path = SOURCE_ANN[split]
        
        if not os.path.exists(ann_path):
            print(f"❌ Skipping {split}: Source annotations not found at {ann_path}")
            continue

        print(f"\n📂 Loading original {split} annotations...")
        with open(ann_path, 'r') as f:
            all_data = json.load(f)

        # Create a dictionary to hold the JSONL entries for each type
        type_buckets = {c_type: [] for c_type in CHART_TYPES}

        print(f"📝 Processing {len(all_data)} entries...")
        for entry in tqdm(all_data, desc=f"Categorizing {split}"):
            chart_type = entry.get('type')
            if chart_type not in type_buckets:
                continue

            # 1. Extract raw data for the summary
            title = entry.get('general_figure_info', {}).get('title', {}).get('text', 'Untitled')
            x_label = entry.get('general_figure_info', {}).get('x_axis', {}).get('label', {}).get('text', 'X-axis')
            y_label = entry.get('general_figure_info', {}).get('y_axis', {}).get('label', {}).get('text', 'Y-axis')
            
            # 2. Build the semantic summary for STEM Sight
            chart_phrase = chart_type.replace('_', ' ')
            summary = f"This is a {chart_phrase} titled '{title}'. The x-axis shows {x_label} and the y-axis shows {y_label}."

            # 3. Format for Donut Training
            type_buckets[chart_type].append({
                "file_name": f"{entry['image_index']}.png",
                "ground_truth": json.dumps({"gt_parse": summary})
            })

        # Save individual JSONL files into the specialized directories
        for c_type in CHART_TYPES:
            output_dir = os.path.join(BASE_PATH, f"{c_type}_plots/{split}")
            output_jsonl = os.path.join(output_dir, "metadata.jsonl")
            
            # Ensure the directory exists (it should from the previous notebook)
            os.makedirs(output_dir, exist_ok=True)

            if type_buckets[c_type]:
                with open(output_jsonl, 'w') as f:
                    for item in type_buckets[c_type]:
                        f.write(json.dumps(item) + "\n")
                print(f"✅ Created {split} metadata for {c_type}: {len(type_buckets[c_type])} entries")

if __name__ == "__main__":
    generate_specialized_metadata()


📂 Loading original train annotations...
📝 Processing 157070 entries...


Categorizing train: 100%|████████████████████████████████████████████| 157070/157070 [00:11<00:00, 13368.50it/s]


✅ Created train metadata for vbar_categorical: 52463 entries
✅ Created train metadata for hbar_categorical: 52700 entries
✅ Created train metadata for dot_line: 26010 entries
✅ Created train metadata for line: 25897 entries

📂 Loading original val annotations...
📝 Processing 33650 entries...


Categorizing val: 100%|███████████████████████████████████████████████| 33650/33650 [00:00<00:00, 192386.68it/s]

✅ Created val metadata for vbar_categorical: 11240 entries


✅ Created val metadata for hbar_categorical: 11292 entries
✅ Created val metadata for dot_line: 5571 entries
✅ Created val metadata for line: 5547 entries


In [1]:
import json

def extract_horizontal_bars(input_jsonl, output_jsonl):
    horizontal_count = 0
    with open(input_jsonl, 'r') as f_in, open(output_jsonl, 'w') as f_out:
        for line in f_in:
            data = json.loads(line)
            # Check for horizontal bar chart types in PlotQA metadata
            if "horizontal_bar" in data.get("type", "").lower():
                f_out.write(json.dumps(data) + "\n")
                horizontal_count += 1
                
    print(f"✅ Extracted {horizontal_count} horizontal bar samples.")

# Run this on your master PlotQA metadata
extract_horizontal_bars("plotqa_master.jsonl", "plotqa_hbar_expert.jsonl")

FileNotFoundError: [Errno 2] No such file or directory: 'plotqa_master.jsonl'